## Settings & imports

In [1]:
import numpy as np
import pandas as pd
import os
import torch
import matplotlib.pyplot as plt
import string
from torch import nn
from torch.utils.data import Dataset, DataLoader
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize
import csv

In [2]:
object_name = 'Konstytucja' # Books / Documents_XVIII_century / Konstytucja

In [3]:
if object_name == 'Books':
    real_or_simulated = 'real' #simulated / real / all
    #comment: in case of classification problem, we skip augmentation, read below why

keep_sample_together = False #Should all 15 slices from the same sample go together to the same set (train/test)?
if keep_sample_together:
    sample_together = 'sample_together'
else:
    sample_together = 'sample_split'
    
preprocessing_method = 'logarithm' # none / normalization / logarithm

columns_to_keep_inks = {
                        'Books': [
                                                 'Al_inks',
                                                 'S_inks',
                                                 'Cr_inks',
                                                 'Mn_inks',
                                                 'Co_inks',
                                                 'Cu_inks',
                                                 'Zn_inks',
                                                 'Pb_inks'],
    
                        'Documents_XVIII_century' : [
                                                 'Al_inks',
                                                 'S_inks',
                                                 'Cr_inks',
                                                 'Mn_inks',
                                                 'Co_inks',
                                                 'Cu_inks',
                                                 'Zn_inks',
                                                 'Pb_inks'
                                                            ],
    
                        'Konstytucja' : [
                                                 'Al',
                                                 'S',
                                                 'Cr',
                                                 'Mn',
                                                 'Co',
                                                 'Cu',
                                                 'Zn',
                                                 'Pb'
                                                            ]
    
                        }

columns_to_keep_inds = { 
                        'Books' : [
                            
                                                 'Al_inds',
                                                 'S_inds',
                                                 'Cr_inds',
                                                 'Mn_inds',
                                                 'Co_inds',
                                                 'Cu_inds',
                                                 'Zn_inds',
                                                 'Pb_inds'],
    
                        'Documents_XVIII_century' : [
                                                 'Al_inds',
                                                 'S_inds',
                                                 'Cr_inds',
                                                 'Mn_inds',
                                                 'Co_inds',
                                                 'Cu_inds',
                                                 'Zn_inds',
                                                 'Pb_inds'
                                                        ],
                        'Konstytucja' : [
                                                'Al_inds',
                                                'S_inds',
                                                'Cr_inds',
                                                'Mn_inds',
                                                'Co_inds',
                                                'Cu_inds',
                                                'Zn_inds',
                                                'Pb_inds'
                        ]
    
                        }

In [4]:
data_path = {'Books' : '../data/DANE.xlsx',
             'Documents_XVIII_century' : '../data/Lipiec_2024.xlsx',
             'Konstytucja' : '../data/Lipiec_2024.xlsx'}

results_path = {'Books' : '../results/Books/',
                'Documents_XVIII_century' : '../results/Documents_XVIII_century/',
                'Konstytucja' : '../results/Konstytucja/'}

figures_path = {'Books' : '../results/visualisations/Books/',
                'Documents_XVIII_century' : '../results/visualisations/Documents_XVIII_century/',
                'Konstytucja': '../results/visualisations/Konstytucja/'}

models_path = {'Books' : '../models/Books/',
                'Documents_XVIII_century' : '../models/Documents_XVIII_century/model_classification_2024_09_09_14_52_15',
                'Konstytucja' : '../models/Documents_XVIII_century/model_classification_2024_09_09_14_52_15'}

## Loading the data

In [5]:
df = pd.ExcelFile(data_path[object_name])

if object_name == 'Books':
    if real_or_simulated == 'simulated':
        inks_df = df.parse('a.', header=0, index_col=0, usecols=list)[:1425] #inKs
        inds_df = df.parse('i.', header=0, index_col=0, usecols=list)[:1425]  #inDs
    elif real_or_simulated == 'real':
        inks_df = df.parse('a.', header=0, index_col=0, usecols=list)[1425:] #inKs
        inds_df = df.parse('i.', header=0, index_col=0, usecols=list)[1425:]  #inDs
    elif real_or_simulated == 'all':
        inks_df = df.parse('a.', header=0, index_col=0, usecols=list) #inKs
        inds_df = df.parse('i.', header=0, index_col=0, usecols=list) #inDs

elif object_name == 'Documents_XVIII_century':
    inks_df = df.parse('Arkusz1', header=0, usecols=range(2,29), skiprows=range(3931, 4742))
    inds_df = df.parse('Arkusz1', header=0, usecols=range(32,59), skiprows=range(3931, 4742))
    inds_df.columns = inks_df.columns
    
elif object_name == 'Konstytucja':
    inds_df = df.parse('Arkusz1', header=0, usecols=range(32,59), skiprows=range(1, 3945))
    inds_df.columns = [x.split('.')[0]+'_inds' for x in list(inds_df.columns)]
    inds_df.drop(index=[0], inplace=True)

In [6]:
inds_df.shape

(796, 27)

## Preprocessing

In [7]:
inds_df = inds_df.reset_index(drop=True)

### Removing some data

1. Let's keep only columns that we need.

To reduce the set of used elements run cell below. Then, instead of predicting 29 numbers, we will predict only 8. We will also use only 8 numbers as input.

In [8]:
inds_df = inds_df[columns_to_keep_inds[object_name]]

2. Let's remove rows with missing values.

In [9]:
(inds_df.shape[0] - inds_df.dropna().shape[0])/inds_df.shape[0]

0.001256281407035176

In [10]:
inds_df.dropna(inplace=True)

In [11]:
X = np.array(inds_df.values)

In [12]:
######################################################

### Normalizing / taking logarithm

In [13]:
def adjusted_log_transform(nonnegative_array):
    res = np.where(nonnegative_array>0, np.log(nonnegative_array), 0.)
    res = np.where(res != 0, res, 2*res.min(axis=0))
    return res

In [14]:
if preprocessing_method == 'normalization':

    X = (X - np.min(X, axis=0))/np.std(X, axis=0)
    
elif preprocessing_method == 'logarithm':
    
    X = adjusted_log_transform(X)
    
elif preprocessing_method == 'none':
    
    pass

/tmp/ipykernel_7214/1324458353.py:2: RuntimeWarning: divide by zero encountered in log
  res = np.where(nonnegative_array>0, np.log(nonnegative_array), 0.)


In [15]:
# coor = 7
# plt.hist(X[:,coor], bins=100)

### Converting to tensors

In [16]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

Using cuda device


In [17]:
X = torch.Tensor(X).to(device)

## Loading the trained model

### Model class

In [18]:
input_size = 8
output_size = 262

class InksNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.seq = nn.Sequential(
        nn.Linear(input_size, input_size),
        nn.ReLU(),
        nn.Linear(input_size, 50),
        nn.ReLU(),
        nn.Linear(50, 100),
        nn.ReLU(),
        nn.Linear(100, output_size))
    def forward(self, x):
        return self.seq(x)

### Loading

In [19]:
model = InksNet()
model.load_state_dict(torch.load(models_path[object_name]))
model = model.to(device)

## Prediction

In [20]:
model.eval()
correct = 0
outputs = model(X)
outputs_class = nn.functional.softmax(outputs).argmax(1)
outputs_class = np.array(outputs_class.cpu())

/tmp/ipykernel_7214/1469060920.py:4: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  outputs_class = nn.functional.softmax(outputs).argmax(1)


### Translating numbers to classes names from the original file

1. Let's create a dictionary.

In [21]:
classes_names_from_file = pd.ExcelFile(data_path[object_name])

if object_name == 'Books':
    pass

elif object_name == 'Documents_XVIII_century':
    pass
    
elif object_name == 'Konstytucja':
    classes_names_from_file = classes_names_from_file.parse('Arkusz1', header=0, usecols=[1], skiprows=range(3931, 4742))
    classes_names_from_file.columns = ['Names from file']

In [22]:
classes_names_from_file['Class number'] = np.repeat(range(int(classes_names_from_file.shape[0]/15)), 15, axis=0)
classes_names_from_file.drop_duplicates(inplace=True)

In [23]:
classes_dict = dict(zip(classes_names_from_file['Class number'], classes_names_from_file['Names from file']))

2. Let's translate numbers to classes names from the file.

In [24]:
outputs_class = list(outputs_class)

In [25]:
outputs_class_translated = [classes_dict[el] for el in outputs_class]

In [26]:
outputs_class_translated

['66.a',
 '40.a',
 '40.a',
 'D62.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 'D62.a',
 'D62.a',
 'D62.a',
 'D62.a',
 'D62.a',
 'D62.a',
 'D62.a',
 'D62.a',
 'D62.a',
 'D62.a',
 'D62.a',
 'D62.a',
 'D62.a',
 'D62.a',
 'D62.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 'D35.a',
 'D35.a',
 'D35.a',
 'D35.a',
 'D35.a',
 'D35.a',
 'D35.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 'E9.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 '40.a',
 'D62.a',
 '40.a',
 'D44.a',
 'D44.a',
 'D44.a',
 'D44.a',
 'D44.a',
 'D44.a',
 'D44.a',
 'D44.a',
 'D39.a',
 'D44.a',
 'D44.a',
 'D44.a',
 'D44.a',
 'D44.a',
 'D44.a',
 '63.a',
 '61.a'

3. Saving result to a file.

In [27]:
model_name = models_path[object_name].split('/')[-1]

In [28]:
with open(results_path[object_name] + 'prediction_from_' + model_name, 'w') as myfile:
    wr = csv.writer(myfile)
    wr.writerow(outputs_class_translated)